# Record model's parametrization process under 3 Hessian Controller options

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import sys, os
sys.path.append('../')
import MeshFEM
import mesh, mesh_energy, viewer, benchmark
import math, bisect, time
import numpy as np
import mesh_operations

In [ ]:
import plot_video_utils

In [ ]:
base_path = 'local_exps_with_uv_2/'

In [ ]:
model_name_list = [entry.name for entry in os.scandir(base_path) if entry.is_dir()]
# delete Figs in model_name_list 
if 'Figs' in model_name_list:  model_name_list.remove('Figs')
if 'Videos' in model_name_list:  model_name_list.remove('Videos')

In [ ]:
thread_num_list = [0]
hessian_option_list = ['Adaptive', 'Always', 'Never']
hessian_option_list = ['Adaptive']

In [ ]:
user_model_name = 'bear_cut'
if user_model_name not in model_name_list:
    raise RuntimeError(f"[Error] '{user_model_name}' is not in directory {base_path} !")

In [ ]:
model_path = f'../../../Models/SmallModels/{user_model_name}.off'

In [ ]:
video_folder = 'Videos'
video_dir = os.path.join(base_path, video_folder)
if not os.path.exists(video_dir):  os.makedirs(video_dir)

In [ ]:
m = mesh.Mesh(model_path)

In [ ]:
fps = 30

In [ ]:
obj_grad_time_list = plot_video_utils.readConvergenceTimingData(os.path.join(base_path, user_model_name), thread_num_list)

In [ ]:
obj_list, grad_norm_list, hessian_projected_list, hessian_shifted_amount_list = plot_video_utils.readHessianData(os.path.join(base_path, user_model_name))

In [ ]:
aligned_timing_list = plot_video_utils.alignTiming(obj_grad_time_list, grad_norm_list)

In [ ]:
for hessian_ind, hessian_option in enumerate(hessian_option_list):
    uv = mesh_energy.NodalVars(m, 2)
    uv_path = os.path.join(base_path, user_model_name, hessian_option, 'UVs')
    numUVs = plot_video_utils.getNumUVs(uv_path)
    uv_data_0 = plot_video_utils.read_uv_data(uv_path, 0)
    uv.setVars(uv_data_0)
    uv_data_final = plot_video_utils.read_uv_data(uv_path, numUVs - 1)
    m_union = mesh_operations.concatenateMeshes([(uv_data_0.reshape(-1, 2), m.elements()), (uv_data_final.reshape(-1, 2), m.elements())])
    
    # create viewer
    v = viewer.Viewer(m_union, wireframe=True)
    em = MeshFEM.EmbeddedMesh(m, uv)
    v.update(mesh=em, preserveExisting=False)
    v.makeOpaque(color='#48B3FF')
    
    spf = 1 / fps
    total_time = aligned_timing_list[hessian_ind][-1]
    numFrames = int(math.ceil(total_time / spf))
    video_fn = user_model_name + '_' + hessian_option + '_symmdsUVopt.mp4'
    
    start_record_timer = time.time()
    v.recordStart(os.path.join(video_dir, video_fn), renderScale=8, outputScale=2, framerate=fps, lineWidthScale=0.25)
    for f in range(numFrames):
        frameTime = f * spf
        iterationForFrame = max(0, bisect.bisect_right(aligned_timing_list[hessian_ind], frameTime) - 1)
        uv.setVars(plot_video_utils.read_uv_data(uv_path, iterationForFrame))
        v.update()
    v.recordStop()
    elapsed_record_time = time.time() - start_record_timer
    print(f"[Viedo] {video_fn} recorded in {video_dir}. Recording Time: {elapsed_record_time:.4f} seconds.")